In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'silver';
CREATE WIDGET TEXT schema_sink DEFAULT 'gold';
CREATE WIDGET TEXT source_sales DEFAULT 'silver_sales';
CREATE WIDGET TEXT sink_sales DEFAULT 'gold_sales';
CREATE WIDGET TEXT w_sales DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_sales = dbutils.widgets.get('source_sales')
sink_sales = dbutils.widgets.get('sink_sales')
w_sales = dbutils.widgets.get('w_sales')

In [0]:
source_sales_table = f'{catalog_name}.{schema_source}.{source_sales}'
df_sales = spark.sql(f"SELECT * FROM {source_sales_table} WHERE updated_at > '{w_sales}'")

In [0]:
df_gold_sales = (
    df_sales
    .withColumn('class_quantity', 
        F.when(F.col('quantity') <= 10, 'top')
        .when(F.col('quantity') <= 50, 'medium').otherwise('low')
    )
    .withColumn('class_salessub', 
        F.when(F.col('sales_subtotal') >= 1000, 'top')
        .when(F.col('sales_subtotal') >= 500, 'medium').otherwise('low')
    )
)

In [0]:
df_gold_sales.printSchema()

In [0]:
target_sink_table = f'{catalog_name}.{schema_sink}.{sink_sales}'
if spark.catalog.tableExists(target_sink_table):

    source_table = DeltaTable.forName(spark, target_sink_table)
    df_gold_sales = df_gold_sales.withColumn('_ingestion_timestamp', F.current_timestamp())

    (
        source_table.alias("target")
        .merge(
            df_gold_sales.alias("source"),
            f"target.sales_id = source.sales_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_gold_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('updated_at','medio_de_pago','sales_subtotal')  # Liquid Clustering
        .saveAsTable(target_sink_table)
    )